# 🐳 Topic 05: Model Packaging & REST APIs with FastAPI & Docker

## 1. Architecture: Serving ML Models over REST
To make a model accessible to external microservices, mobile apps, or web frontends, we wrap it in a lightweight REST API (FastAPI) and containerize it (Docker).

---

## 2. Hands-on: Building and Testing a FastAPI Model Server


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
import numpy as np
from sklearn.ensemble import RandomForestClassifier

X_dummy = np.random.rand(100, 4)
y_dummy = np.random.choice([0, 1], size=100)
model = RandomForestClassifier().fit(X_dummy, y_dummy)

app = FastAPI(title="MLOps Prediction API", version="1.0.0")

class PredictionInput(BaseModel):
    features: list[float] = Field(..., example=[0.5, 0.2, 0.8, 0.1], description="4-dimensional feature vector")

class PredictionOutput(BaseModel):
    prediction: int
    probability: float

@app.get("/health")
def health_check():
    return {"status": "healthy", "model": "RandomForestClassifier"}

@app.post("/predict", response_model=PredictionOutput)
def predict(payload: PredictionInput):
    input_vector = np.array(payload.features).reshape(1, -1)
    pred = int(model.predict(input_vector)[0])
    prob = float(model.predict_proba(input_vector)[0][pred])
    return PredictionOutput(prediction=pred, probability=prob)

client = TestClient(app)

res_health = client.get("/health")
print("Health Response:", res_health.json())

res_predict = client.post("/predict", json={"features": [0.5, 0.2, 0.8, 0.1]})
print("Predict Response:", res_predict.json())


---

## 3. Dockerfile Template for ML Microservices

Below is a production-grade `Dockerfile` for serving this FastAPI application:

```dockerfile
FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```
